In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# **Download Data**

In [ ]:
!cp ./gdrive/MyDrive/bracs/train.zip ./train.zip
!unzip train.zip
!rm train.zip
!rm -r ./train/1
!rm -r ./train/2
!rm -r ./train/4
!rm -r ./train/6

In [ ]:
!cp ./gdrive/MyDrive/bracs/bracs_in_test.zip ./bracs_in_test.zip
!unzip bracs_in_test.zip
!rm bracs_in_test.zip
!rm -r ./test/1
!rm -r ./test/2
!rm -r ./test/4
!rm -r ./test/6

# **Download Code**

In [ ]:
!pip install lightly

# **Ready for Train**

In [5]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm import tqdm
import json

from lightly.data import LightlyDataset
from lightly.transforms import  utils
from lightly.transforms.vicreg_transform import VICRegTransform

In [6]:
num_workers = 2
batch_size = 32
seed = 1
max_epochs = 2
input_size = 224

In [7]:
pl.seed_everything(seed)

INFO:lightning_fabric.utilities.seed:Seed set to 1


1

In [8]:
path_to_data_train = "./train"
path_to_data_test = "./test"

In [9]:
transform = VICRegTransform(input_size=input_size)

test_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((input_size, input_size)),
        torchvision.transforms.ToTensor(),
    ]
)

dataset_train_vicreg = LightlyDataset(input_dir=path_to_data_train, transform=transform)

dataset_test = LightlyDataset(input_dir=path_to_data_test, transform=test_transform)

dataloader_train_vicreg = torch.utils.data.DataLoader(
    dataset_train_vicreg,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers,
)

dataloader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [10]:
from lightly.loss.vicreg_loss import VICRegLoss
from lightly.models.modules.heads import VICRegProjectionHead


class VICRegModel(pl.LightningModule):
    def __init__(self):
        super().__init__()

        # https://discuss.pytorch.org/t/feature-extraction-in-torchvision-models-vit-b-16/148029
        self.vit = torchvision.models.vit_b_16(pretrained=True)
        self.backbone = nn.Sequential(*list(self.vit.children())[:-1])
        self.encoder = self.backbone[1]

        hidden_dim = self.vit.heads.head.in_features
        self.projection_head = VICRegProjectionHead(
            input_dim=hidden_dim,
            hidden_dim=2048,
            output_dim=2048,
            num_layers=2,
        )
        self.criterion = VICRegLoss()

    def forward(self, x):
        x = self.vit._process_input(x)
        n = x.shape[0]
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.encoder(x)
        x = x[:, 0]

        z = self.projection_head(x)
        return z

    def get_embedding(self, x):
        x = self.vit._process_input(x)
        n = x.shape[0]
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = self.encoder(x)
        x = x[:, 0]

        return x

    def training_step(self, batch, batch_idx):
        (x0, x1), _, _ = batch
        z0 = self.forward(x0)
        z1 = self.forward(x1)
        loss = self.criterion(z0, z1)
        self.log("train_loss_ssl", loss)
        return loss

    def configure_optimizers(self):
        optim = torch.optim.SGD(
            self.parameters(), lr=0.0001, momentum=0.9, weight_decay=5e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, max_epochs)
        return [optim], [scheduler]

In [11]:
model = VICRegModel()

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:04<00:00, 75.1MB/s]


if you have a checkpoint to load, give the path here and uncomment the code

In [12]:
# checkpoint_path = "path/to/checkpoint"
# ckpt = torch.load(checkpoint_path)


# model.load_state_dict(ckpt['state_dict'])

In [13]:
trainer = pl.Trainer(max_epochs=max_epochs, devices=1, accelerator="gpu")
trainer.fit(model, dataloader_train_vicreg)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type                 | Params
---------------------------------------------------------
0 | vit             | VisionTransformer    | 86.6 M
1 | backbone        | Sequential           | 85.8 M
2 | encoder         | Encoder              | 85.2 M
3 | projection_head | VICRegProjectionHead | 5.8 M 
4 | criterion       | VICRegLoss           | 0     
---------------------------------------------------------
92.3 M    Trainable params
0         Non-trainable params
92.3 M    Total params
369.364   Total estimated model para

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


# **Get Embedding**

In [14]:
new_model = VICRegModel()
checkpoint_path = "./lightning_logs/version_0/checkpoints/epoch=1-step=1858.ckpt"
ckpt = torch.load(checkpoint_path)
new_model.load_state_dict(ckpt['state_dict'])

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<All keys matched successfully>

In [15]:
if torch.cuda.is_available():
  new_model.to(torch.device("cuda"))

In [16]:
def generate_embeddings(model, dataloader):
    """Generates representations for all images in the dataloader with
    the given model
    """

    embeddings = []
    targets = []
    filenames = []
    with torch.no_grad():
        for img, target, fnames in tqdm(dataloader):
            img = img.to(model.device)
            emb = model.get_embedding(img).flatten(start_dim=1)
            embeddings.append(emb)
            targets.append(target)
            filenames.extend(fnames)

    embeddings = torch.cat(embeddings, 0)
    targets = torch.cat(targets, 0)
    return embeddings, targets, filenames


new_model.eval()
embeddings, targets, filenames = generate_embeddings(new_model, dataloader_test)

100%|██████████| 94/94 [00:32<00:00,  2.87it/s]


In [17]:
print(embeddings.shape)
print(targets.shape)
print(len(filenames))

torch.Size([3000, 768])
torch.Size([3000])
3000


In [18]:
torch.save(embeddings, "./embeddings.pth")
torch.save(targets, "./labels.pth")
all_file_paths = {"file_names": filenames}
with open("./file_names.json", 'w') as f2:
  json.dump(all_file_paths, f2)